In [4]:
import os
import time
import torch
import numpy as np
import torch.nn.functional as F
from tqdm import tqdm
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import Planetoid, Coauthor, CitationFull
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [2]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class Net2(torch.nn.Module):
    def __init__(self, num_features, hidden, num_layers, num_classes):
        super(Net2, self).__init__()
        self.num_layers = num_layers
        self.conv = torch.nn.ModuleList()
        self.conv.append(GCNConv(num_features, hidden))
        for i in range(self.num_layers - 1):
            self.conv.append(GCNConv(hidden, hidden))
        self.lt1 = torch.nn.Linear(hidden, num_classes)

    def reset_parameters(self):
        for module in self.conv:
            module.reset_parameters()
        self.lt1.reset_parameters()

    def forward(self, x, edge_index):
        for i in range(self.num_layers):
            x = self.conv[i](x, edge_index)
            x = F.elu(x)
            x = F.dropout(x, training=self.training)
        x = self.lt1(x)
        return F.log_softmax(x, dim = 1)
    
def index_to_mask(index, size):
    mask = torch.zeros(size, dtype=torch.bool, device=index.device)
    mask[index] = 1
    return mask

def splits_classification(data, num_classes, exp):
    if exp!='fixed':
        indices = []
        for i in range(num_classes):
            index = (data.y == i).nonzero().view(-1)
            index = index[torch.randperm(index.size(0))]
            indices.append(index)

        if exp == 'random':
            train_index = torch.cat([i[:20] for i in indices], dim=0)
            val_index = torch.cat([i[20:50] for i in indices], dim=0)
            test_index = torch.cat([i[50:] for i in indices], dim=0)
        else:
            train_index = torch.cat([i[:5] for i in indices], dim=0)
            val_index = torch.cat([i[5:10] for i in indices], dim=0)
            test_index = torch.cat([i[10:] for i in indices], dim=0)

        data.train_mask = index_to_mask(train_index, size=data.num_nodes)
        data.val_mask = index_to_mask(val_index, size=data.num_nodes)
        data.test_mask = index_to_mask(test_index, size=data.num_nodes)

    return data

In [7]:
if not os.path.exists("../save"):
  os.mkdir("../save")
if not os.path.exists("../save/node_cls"):
  os.mkdir("../save/node_cls")
if not os.path.exists("../save/node_cls/baselines"):
  os.mkdir("../save/node_cls/baselines")

dataset_names = ['cora']
for dataset_name in dataset_names:
    dataset = Planetoid(root='../dataset', name=dataset_name)
    num_classes = dataset.num_classes
    data = splits_classification(dataset[0], num_classes, exp = 'fixed')
    model = Net2(data.x.shape[1], 512, 2, num_classes).to(device)
    loss_fn = torch.nn.NLLLoss()
    model.reset_parameters()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    avg_time = 0
    all_acc = []
    for run in range(5):
        best_val_loss = 100000
        model.reset_parameters()

        data = data.to(device)
        for epoch in tqdm(range(300)):
            model.train()
            optimizer.zero_grad()
            out = model(data.x, data.edge_index)
            loss = loss_fn(out[data.train_mask], data.y[data.train_mask])
            loss.backward()
            optimizer.step()

            model.eval()
            with torch.no_grad():
                out = model(data.x, data.edge_index)
                val_loss = loss_fn(out[data.val_mask], data.y[data.val_mask])
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    #save model
                    torch.save(model.state_dict(), f'../save/node_cls/baselines/baseline_{dataset_name}_fixed.pt')
        start = time.time()
        out = model(data.x, data.edge_index)
        avg_time += time.time() - start
        test_loss = loss_fn(out[data.test_mask], data.y[data.test_mask])
        acc = int(torch.sum(torch.argmax(out, dim=1) == data.y).item()) / len(data.y)
        all_acc.append(acc)
    top_acc= sorted(all_acc, reverse = True)[:10]
    print(f'{dataset_name} fixed: {np.mean(all_acc):.4f} ± {np.std(all_acc):.4f} time: {avg_time/5:.4f}')

100%|██████████| 300/300 [00:03<00:00, 98.88it/s] 

cora fixed: 0.8021 ± 0.0159 time: 0.0031
